# 玩家風格嵌入（Model B）訓練與推論

執行 `aicup.train_player_emb` 與 `aicup.infer_player_emb`。checkpoint 存到 Drive 的 `player-trained-models/<名稱>/`，嵌入快取存到 checkpoint 旁的 `emb-cache/`。

- 系統 RAM 要 30 GB 以上
- 依序執行全部 cell；訓練中斷後重跑同一個 MODE 會從 `last.pt` 接續，接續時 `COMMIT` 要和開始時相同

| MODE | 內容 | 競賽一最終版本 |
|---|---|---|
| `model_b` | 以 `rank-cnn-v180000` 初始化，512 維嵌入、尾盤截斷 40 手、seed 1024、35 epoch | Model B |
| `infer_b` | B 組候選與題目的嵌入，8 種 D4 TTA | Model B 的 B 組快取 |
| `infer_a` | A 組候選與題目的嵌入，TTA 數為 `INFER_TTA_A` | Model B 的 A 組快取 |
| `infer_val`、`infer_hold` | newdev、val648、封存組的嵌入，供本地評估 | 本地評估 |
| `smoke` | 2 epoch，確認環境 | |
| `full`、`stage2` | 預設嵌入維度的訓練與第二階段微調 | 不在最終版本流程內 |

- `HEAD_MOVES = 249` 時 `infer_*` 改產生只看前段手數的快取 `*_head_emb.npz`，最終版本不用。

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下，在本機產生後上傳。

| 路徑 | MODE | 產生方式（本機，repo 根目錄） |
|---|---|---|
| `rank-features-v180000/train/`、`val/` | 訓練、`infer_val`、`infer_hold` | `python -m aicup.extract_features --train-dir training-v1.1/training --val-size 180000 --out rank-features-v180000` |
| `rank-features-v180000/hd.npz` | 訓練、`infer_val`、`infer_hold` | `python -m aicup.color --train-dir training-v1.1/training --out rank-features-v180000/hd.npz` |
| `rank-trained-models/rank-cnn-v180000/best_model.pth` | `smoke`、`full`、`model_b` | `colab_train.ipynb` 的 `rank_cnn_v180000` |
| `humansl-analysis/val-players.csv`、`val-players-newdev-v180000.csv` | 訓練、`infer_val` | `python -m aicup.val_players --val-size 180000 --out-dir humansl-analysis` |
| `humansl-analysis/holdout-v180000.csv` | `infer_hold` | 同上 |
| `training-v1.1/training/train_*.csv` | `HEAD_MOVES` 不為 0 的 `infer_val` | 官方訓練集 v1.1，見 README |
| `tests/player_identification_*.csv` | `infer_a`、`infer_b` | AI CUP 競賽平台 |
| `INFER_CKPT` | `infer_*` | 這份 notebook 的 `model_b` |
| `RESUME_CKPT` | `stage2` | 這份 notebook 的 `full` |

## 參數

In [ ]:
import os

REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
MODE = 'smoke'
FEAT_DIR = 'rank-features-v180000'
INIT_CKPT = 'rank-trained-models/rank-cnn-v180000/best_model.pth'
RESUME_CKPT = 'player-trained-models/emb-full/last.pt'  # stage2 用
INFER_CKPT = 'player-trained-models/emb-model-b/best_model.pth'  # infer_* 用
INFER_TTA_A = 4  # A 組推論的 D4 視角數
HEAD_MOVES = 0  # 前段視角填 249（加根節點空盤正好 250 個盤面），0 為完整視角
EXTRA_ARGS = []
AUTO_UNASSIGN = True

TRAIN = ['python', '-m', 'aicup.train_player_emb', '--feat-dir', FEAT_DIR]
INFER = ['python', '-m', 'aicup.infer_player_emb', '--ckpt', INFER_CKPT]
COMMON = ['--steps-per-epoch', '1500', '--color', 'swap']
CACHE_DIR = os.path.dirname(INFER_CKPT) + '/emb-cache'
SUFFIX = '_head_emb.npz' if HEAD_MOVES else '_emb.npz'
HEAD = ['--head-moves', str(HEAD_MOVES)] if HEAD_MOVES else []
NEWDEV, VAL648 = 'humansl-analysis/val-players-newdev-v180000.csv', 'humansl-analysis/val-players.csv'
HOLD = 'humansl-analysis/holdout-v180000.csv'
RUNS = {  # MODE: 依序執行的指令
    'smoke': [TRAIN + ['--init', INIT_CKPT, '--ckpt-dir', 'player-trained-models/emb-smoke', '--epochs', '2', *COMMON]],
    'full': [TRAIN + ['--init', INIT_CKPT, '--ckpt-dir', 'player-trained-models/emb-full', '--epochs', '20', *COMMON]],
    'stage2': [TRAIN + ['--resume-weights', RESUME_CKPT, '--ckpt-dir', 'player-trained-models/emb-stage2',
                        '--epochs', '15', '--steps-per-epoch', '1500', '--lr', '3e-4', '--backbone-lr', '1e-4',
                        '--tail-crop', '20', '--seed', '142', '--color', 'swap']],
    'model_b': [TRAIN + ['--init', INIT_CKPT, '--ckpt-dir', 'player-trained-models/emb-model-b', '--embed-dim', '512',
                         '--tail-crop', '40', '--seed', '1024', '--epochs', '35', *COMMON]],
    'infer_val': [INFER + ['--mode', 'packed', '--feat-dir', FEAT_DIR, '--players', players, '--tta', '8',
                           '--batch-size', '128', '--out', CACHE_DIR + '/' + tag + SUFFIX,
                           *(HEAD + ['--sgf-dir', 'training-v1.1/training'] if HEAD_MOVES else [])]
                  for tag, players in (('newdev', NEWDEV), ('val648', VAL648))],
    'infer_hold': [INFER + ['--mode', 'packed', '--feat-dir', FEAT_DIR, '--players', HOLD, '--tta', '8',
                            '--batch-size', '128', '--out', CACHE_DIR + '/hold_emb.npz']],
    'infer_b': [INFER + ['--mode', 'test', '--groups', 'b', '--tta', '8', '--batch-size', '64',
                         '--out-dir', CACHE_DIR, *HEAD]],
    'infer_a': [INFER + ['--mode', 'test', '--groups', 'a', '--tta', str(INFER_TTA_A), '--batch-size', '64',
                         '--out-dir', CACHE_DIR, *HEAD]],
}
if MODE not in RUNS:
    raise SystemExit('MODE 要是 %s 之一' % ', '.join(RUNS))
CMDS = [cmd + EXTRA_ARGS for cmd in RUNS[MODE]]

FEAT_HINT = ('python -m aicup.extract_features --train-dir training-v1.1/training --val-size 180000 '
             '--out rank-features-v180000')
LIST_HINT = 'python -m aicup.val_players --val-size 180000 --out-dir humansl-analysis'
TEST_HINT = 'AI CUP 競賽平台'
NEEDS = []
if MODE in ('smoke', 'full', 'stage2', 'model_b', 'infer_val', 'infer_hold'):
    NEEDS += [(FEAT_DIR + '/train', FEAT_HINT), (FEAT_DIR + '/val', FEAT_HINT),
              (FEAT_DIR + '/hd.npz', 'python -m aicup.color --train-dir training-v1.1/training --out %s/hd.npz' % FEAT_DIR)]
if MODE in ('smoke', 'full', 'stage2', 'model_b', 'infer_val'):
    NEEDS += [(NEWDEV, LIST_HINT), (VAL648, LIST_HINT)]
if MODE in ('smoke', 'full', 'model_b'):
    NEEDS.append((INIT_CKPT, 'colab_train.ipynb 的 MODE rank_cnn_v180000'))
if MODE == 'stage2':
    NEEDS.append((RESUME_CKPT, '這份 notebook 的 MODE full'))
if MODE.startswith('infer_'):
    NEEDS.append((INFER_CKPT, '這份 notebook 的 MODE model_b'))
if MODE == 'infer_hold':
    NEEDS.append((HOLD, LIST_HINT))
if MODE == 'infer_val' and HEAD_MOVES:
    NEEDS.append(('training-v1.1/training', '官方訓練集 v1.1，見 README'))
for group, suffix in (('infer_a', ''), ('infer_b', '2')):
    if MODE == group:
        NEEDS += [('tests/player_identification_%s%s_%s.csv' % (kind, suffix, split), TEST_HINT)
                  for kind in ('test', 'candidates') for split in ('public', 'private')]
for cmd in CMDS:
    print(' '.join(cmd))

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi -L

## 準備資料

要看到：Drive 目錄的連結、需要的檔案都在、系統 RAM。訓練的 checkpoint 目錄已用別的 commit 訓練過時停止。

In [ ]:
link_drive([FEAT_DIR, 'rank-trained-models', 'player-trained-models', 'humansl-analysis', 'tests', 'training-v1.1'])
need(NEEDS)
if not MODE.startswith('infer_'):
    check_commit(CMDS[0][CMDS[0].index('--ckpt-dir') + 1], GIT_HEAD)
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024 ** 3
print('系統 RAM %.1f GB' % ram_gb)

## 執行

輸出同時寫到 Drive 的 `player-trained-models/logs/<MODE>.log`。

In [ ]:
for cmd in CMDS:
    run(cmd, os.path.join('player-trained-models', 'logs', MODE + '.log'))

## 檢查

要看到：訓練 MODE 印出 `history.csv` 的最後幾個 epoch，推論 MODE 列出 `emb-cache/` 的檔案。

In [ ]:
import pandas as pd

if MODE.startswith('infer_'):
    for name in sorted(os.listdir(CACHE_DIR)):
        print('%-28s %8.1f MB' % (name, os.path.getsize(os.path.join(CACHE_DIR, name)) / 1024 ** 2))
else:
    ckpt_dir = CMDS[0][CMDS[0].index('--ckpt-dir') + 1]
    display(pd.read_csv(os.path.join(ckpt_dir, 'history.csv')).tail().round(4))

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')